In [ ]:
import pandas as pd
import numpy as np

# Load the BTS dataset — this may take 30-60 seconds for 7 million rows
print('Loading BTS flight data...')
df = pd.read_csv('../data/raw/bts_flights_2025.csv', low_memory=False)
 
# Always start every new dataset with these four commands
print('Shape:', df.shape)           # (rows, columns)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isnull().sum())
df.head()

#Filter to the six major U.S. carriers

#IATA codes: UA=United, DL=Delta, AA=American, WN=Southwest, AS=Alaska, B6=JetBlue
major_carriers = ['UA', 'Dl', 'AA', 'WN', 'AS', 'B6']
df = df[df['Reporting_Airline'].isin(major_carriers)].copy()
print(f'Rows after filtering to major carriers: {len(df):,}')

#Map carrier codes to full airline names - much cleaner for charts
airline_names = {
    'UA': 'United Airlines',
    'DL': 'Delta Airlines',
    'AA': 'American Airlines',
    'WN': 'Southwest Airlines',
    'AS': 'Alaska Airlines',
    'B6': 'JetBlue Airways'
}
df['airline_name] = df['Reporting_Airline'].map(airline_names)

#Fix the flight date column - convert text to proper datetime
df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')

#Extract time components for analysis
df['month']         = df['FlightDate'].dt.month
df['month_name']    = df['FlightDate'].dt.strftime('%B')
df['day_of_week']   = df['FlightDate'].dt.day_name()
df['quarter']       = df['FlightDate'].dt.quarter

print('Date range:', df['FlightDate'].min(), 'to', df['FlightDate'].max())
print('Flights per airline:')
print(df['airline_name'].value_counts())


#Handle cancelled flights - they have no delay data
#Separate them for cancellation analysis before filling nulls
df_cancelled = df[df['Cancelled] == 1].copy()
df_operated = df[df['Cancelled] == 0].copy()

print(f'Total flights: {len(df):,}')
print(f'Operated: {len(df_operated:,)} ({len(df_operated)/len(df)*100:.1f}%)')
print(f'Cancelled: {len(df_cancelled:,)} ({len(df_cancelled)/len(df)*100:.1f}%)')

#For operated flights, fill NaN delay causes with 0
#(NaN in delay columns = no delay from that cause)
delay_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
df_operated[delay_cols] = df_operated[delay_cols].fillna(0)
 
# Fill ArrDelay and DepDelay nulls with 0 for on-time flights
df_operated['ArrDelay'] = df_operated['ArrDelay'].fillna(0)
df_operated['DepDelay'] = df_operated['DepDelay'].fillna(0)
 
print('\nRemaining nulls in operated flights:')
print(df_operated[['ArrDelay','DepDelay']+delay_cols].isnull().sum())

# On-time flag: FAA defines on-time as arriving within 15 minutes of schedule
df_operated['on_time'] = df_operated['ArrDelay'] <= 15
 
# Delay severity category
def delay_severity(mins):
    if mins <= 15:   return 'On Time'
    elif mins <= 45: return 'Minor Delay'
    elif mins <= 120:return 'Moderate Delay'
    elif mins <= 240:return 'Severe Delay'
    else:            return 'Extreme Delay (4hr+)'
 
df_operated['delay_severity'] = df_operated['ArrDelay'].apply(delay_severity)
 
# Primary delay cause — which cause contributed the most minutes?
def primary_cause(row):
    causes = {
        'Carrier':      row['CarrierDelay'],
        'Weather':      row['WeatherDelay'],
        'NAS':          row['NASDelay'],
        'Security':     row['SecurityDelay'],
        'Late Aircraft':row['LateAircraftDelay']
    }
    max_cause = max(causes, key=causes.get)
    return max_cause if causes[max_cause] > 0 else 'No Delay'
 
df_operated['primary_delay_cause'] = df_operated.apply(primary_cause, axis=1)
 
# Distance tier for route analysis
df_operated['distance_tier'] = pd.cut(
    df_operated['Distance'],
    bins=[0, 500, 1000, 2000, 9999],
    labels=['Short (<500mi)','Medium (500-1000mi)','Long (1000-2000mi)','Ultra Long (2000mi+)']
)
 
print('Business metrics created successfully')
print('On-time rate:', round(df_operated['on_time'].mean()*100, 2), '%')
print('\nDelay severity breakdown:')
print(df_operated['delay_severity'].value_counts())

# Export the main operated flights dataset
df_operated.to_csv('../data/cleaned/flights_cleaned.csv', index=False)
print(f'Operated flights saved: {len(df_operated):,} rows, {len(df_operated.columns)} columns')
 
# Export cancelled flights separately
df_cancelled.to_csv('../data/cleaned/flights_cancelled.csv', index=False)
print(f'Cancelled flights saved: {len(df_cancelled):,} rows')
 
# Print a cleaning summary for your Markdown cell
print(f'\nCleaning Summary:')
print(f'Original rows:  {len(df):,}')
print(f'Operated rows:  {len(df_operated):,}')
print(f'Cancelled rows: {len(df_cancelled):,}')
print(f'Carriers: {df_operated["airline_name"].nunique()}')
print(f'Airports: {df_operated["Origin"].nunique()}')

